# 1. The data and the folds

This notebook builds the splits everything else depends on. It is the one place
where a silent mistake would invalidate every later result, so each step is
checked rather than assumed.

**What we have.** Three datasets, all with the same shape: subjects enter at
different calendar times, each is followed for a fixed 72-month window, and each ends in
one of several mutually exclusive events (or none).

| | subjects | entry cohort | competing events |
|---|---|---|---|
| financial, calm | 55,000 loans | origination year 2013–2018 | default / prepayment |
| financial, crisis | 55,000 loans | origination year 2005–2009 | default / prepayment |
| medical | ~43,700 people | survey cycle 1999–2014 | cardiovascular / cancer / other death |

In [1]:
import os, sys, json
# notebooks run from notebooks/, but every path in this project is relative to
# the project root, so step up once if that is where we are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working directory:", os.getcwd().split("/")[-1])

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
BANK = "data/banking/1_merged"
MED  = "data/medical/1_merged"

calm   = pd.read_csv(f"{BANK}/financial_calm_2013_2018.csv")
crisis = pd.read_csv(f"{BANK}/financial_crisis_2005_2009.csv")
print("calm  ", calm.shape)
print("crisis", crisis.shape)
calm.head(3)

working directory: project


calm   (55000, 22)
crisis (55000, 22)


,loan_id,vintage,time,event,default_zb_alt,fico,ltv,cltv,dti,dti_missing,...,orig_term,mi_pct,n_units,n_borrowers,first_time_buyer,occupancy,channel,loan_purpose,property_type,state
0,F13Q10000078,2013,55,2,0,737.0,117.0,117.0,NaN,1,...,360,0,1,2,N,P,B,N,SF,CA
1,F13Q10000788,2013,72,0,0,669.0,142.0,142.0,NaN,1,...,360,0,1,2,N,P,B,N,SF,NV
2,F13Q10001101,2013,72,0,0,811.0,59.0,59.0,33.0,0,...,180,0,1,1,N,P,R,C,SF,MI


## 1.1 What `time` and `event` mean

These two columns are the outcome. Everything else is a predictor.

- **`time`** — one number per subject: the month at which we stopped learning
  anything about them. Either the event happened, or the 72-month window closed.
- **`event`** — what happened at that moment. `0` means nothing happened and the
  subject was still being observed (**censored**); `1` and `2` are the two
  competing events.

A censored row is **not** missing data. It says *this loan survived at least 72
months*, which is real information. Roughly 35% of the financial rows and 93% of
the medical rows are censored, so discarding them would discard the study.

In [2]:
for name, d in (("calm", calm), ("crisis", crisis)):
    vc = d.event.value_counts().reindex([0,1,2]).fillna(0).astype(int)
    print(f"{name:7s}  censored {vc[0]:>7,}  default {vc[1]:>6,}  prepaid {vc[2]:>7,}")

calm     censored  19,458  default  2,184  prepaid  33,358
crisis   censored  14,348  default  4,793  prepaid  35,859


## 1.2 Why there are two financial cohorts

The hypothesis under test is that the evaluation protocol matters *because the
population shifts between training and test*. That predicts a bigger effect when
the shift is bigger — so the two cohorts manipulate the size of the shift and
hold everything else fixed.

Look at the default rate by origination year below. The calm cohort drifts
monotonically over about a 3× range. The crisis cohort swings over a 7× range
**and reverses**: 2009 loans default less often than 2005 loans, because they
were underwritten after the collapse to much tighter standards.

In [3]:
for name, d in (("calm", calm), ("crisis", crisis)):
    r = d.groupby("vintage").event.apply(lambda s: (s == 1).mean())
    print(f"{name}:  " + "   ".join(f"{y} {v:.1%}" for y, v in r.items()))

calm:  2013 1.9%   2014 2.7%   2015 4.0%   2016 4.3%   2017 5.5%   2018 5.4%
crisis:  2005 7.8%   2006 11.6%   2007 14.1%   2008 8.1%   2009 2.0%


## 1.3 The two split arms

Each model is evaluated twice on the same data, under two different rules for
assigning subjects to training and test.

**Time-ordered.** Train on earlier entry cohorts, test on the next one. This is
how a model is actually used: a lender scores loans that do not exist yet.

**Random.** Shuffle and split. Training then contains subjects from the *same*
cohorts as the test set — information no deployed model ever has. This is the
protocol most of the benchmarking literature uses.

Neither is wrong. They answer different questions, and the **difference between
them** is the finding.

### Why both arms must train on the same number of rows

An expanding time-ordered window trains on 17% of the data in its first fold and
83% in its last. A plain 5-fold split trains on a constant 80%. Compare those
directly and any difference is confounded with *how much data each model saw* —
not with the split rule, which is the thing under test.

So every time-ordered fold gets a random partner **matched on both training-set
and test-set size**.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

d = calm.sort_values("vintage").reset_index(drop=True)
print("sklearn TimeSeriesSplit splits on ROW COUNT, not on year:\n")
for i, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5).split(d), 1):
    trv, tev = set(d.vintage[tr]), set(d.vintage[te])
    leak = sorted(trv & tev)
    print(f"  fold {i}: train {sorted(trv)}  test {sorted(tev)}   "
          f"{'LEAK ' + str(leak) if leak else 'clean'}")

sklearn TimeSeriesSplit splits on ROW COUNT, not on year:

  fold 1: train [2013, 2014]  test [2014, 2015]   LEAK [2014]
  fold 2: train [2013, 2014, 2015]  test [2015, 2016]   LEAK [2015]
  fold 3: train [2013, 2014, 2015, 2016]  test [2016, 2017]   LEAK [2016]
  fold 4: train [2013, 2014, 2015, 2016, 2017]  test [2017]   LEAK [2017]
  fold 5: train [2013, 2014, 2015, 2016, 2017]  test [2018]   clean


Four of the five folds put part of a vintage in training and the rest of the
**same vintage** in test. That is exactly the leakage the time-ordered arm exists
to prevent — so we do not use `TimeSeriesSplit`. We split on cohort boundaries
instead, which cannot leak by construction.

In [5]:
sys.path.insert(0, "src")
import importlib
make_folds = importlib.import_module("1_make_folds")
temporal_folds = make_folds.temporal_folds
matched_random_folds = make_folds.matched_random_folds
verify = make_folds.verify

groups = d.vintage.to_numpy()
events = d.event.to_numpy()

tf = temporal_folds(groups)
rf = matched_random_folds(groups, events, tf, seed=0)
ok = verify("calm cohort", groups, events, tf, rf)
print("\nverified:", ok)


=== calm cohort ===
  fold    train    test  train groups                 test group    events in test
     1    9,166   9,166  2013                               2014  0:3661 1:246 2:5259
     2   18,332   9,167  2013-2014                          2015  0:3307 1:370 2:5490
     3   27,499   9,168  2013-2015                          2016  0:3110 1:391 2:5667
     4   36,667   9,167  2013-2016                          2017  0:2478 1:508 2:6181
     5   45,834   9,166  2013-2017                          2018  0:1863 1:493 2:6810
  all checks passed: no group leakage, sizes matched, no row in both sides

verified: True


`verify` asserts every property the design depends on, before any model is
fitted:

- no cohort appears on both sides of a time-ordered fold
- no row appears in both training and test
- training never contains a cohort at or after the test cohort
- each random fold matches its time-ordered partner on train and test size

A fold that is silently wrong produces results that look entirely reasonable and
answer a different question, so these are checked rather than trusted.

## 1.4 Build and save the folds

Folds are built **once** and written to disk. Every model reads the same file.
If each model re-split the data, comparisons between models would not be paired
and the bootstrap in notebook 4 would be invalid.

In [6]:
!{sys.executable} src/1_make_folds.py


=== financial-calm ===
  fold    train    test  train groups                 test group    events in test
     1    9,166   9,166  2013                               2014  0:3661 1:246 2:5259
     2   18,332   9,167  2013-2014                          2015  0:3307 1:370 2:5490
     3   27,499   9,168  2013-2015                          2016  0:3110 1:391 2:5667
     4   36,667   9,167  2013-2016                          2017  0:2478 1:508 2:6181
     5   45,834   9,166  2013-2017                          2018  0:1863 1:493 2:6810
  all checks passed: no group leakage, sizes matched, no row in both sides



=== financial-crisis ===
  fold    train    test  train groups                 test group    events in test
     1   11,000  11,000  2005                               2006  0:2805 1:1275 2:6920
     2   22,000  11,000  2005-2006                          2007  0:2141 1:1547 2:7312
     3   33,000  11,000  2005-2007                          2008  0:1638 1:890 2:8472
     4   44,000  11,000  2005-2008                          2009  0:3183 1:219 2:7598
  all checks passed: no group leakage, sizes matched, no row in both sides



=== medical ===
  fold    train    test  train groups                 test group    events in test
     1    4,873   5,400  1999-2000                     2001-2002  0:4936 1:146 2:109 3:209
     2   10,273   5,029  1999-2000-2001-2002           2003-2004  0:4548 1:175 2:110 3:196


     3   15,302   4,977  1999-2000-2003-2004           2005-2006  0:4580 1:137 2:83 3:177
     4   20,279   5,926  1999-2000-2005-2006           2007-2008  0:5430 1:155 2:111 3:230
     5   26,205   6,201  1999-2000-2007-2008           2009-2010  0:5758 1:131 2:118 3:194
     6   32,406   5,545  1999-2000-2009-2010           2011-2012  0:5129 1:123 2:104 3:189
     7   37,951   5,757  1999-2000-2011-2012           2013-2014  0:5330 1:151 2:90 3:186
  all checks passed: no group leakage, sizes matched, no row in both sides

wrote results/folds/financial-calm.json  (5 folds per arm)

wrote results/folds/financial-crisis.json  (4 folds per arm)

wrote results/folds/medical.json  (7 folds per arm)


In [7]:
spec = json.load(open("data/banking/2_folds/financial-calm.json"))
print("saved keys:", list(spec))
print("folds per arm:", len(spec["temporal"]))
f0 = spec["temporal"][0]
print(f"fold 1 -> train {len(f0['train']):,} rows, test {len(f0['test']):,} rows")
print("first 5 training row indices:", f0["train"][:5])

saved keys: ['name', 'group_col', 'seed', 'n', 'temporal', 'random']
folds per arm: 5
fold 1 -> train 9,166 rows, test 9,166 rows
first 5 training row indices: [0, 1, 2, 3, 4]


## What to take from this notebook

1. `time` and `event` are the outcome; censored rows are informative, not missing.
2. Two financial cohorts manipulate the size of the distribution shift.
3. Two split arms answer different questions; the difference between them is the result.
4. `TimeSeriesSplit` leaks across cohort boundaries — we split on the boundaries ourselves.
5. Folds are built once, verified, and reused, so every comparison is paired.

**Next:** notebook 2 fits one model in one fold and computes all four metrics,
so the API shapes and the metric meanings are clear before running the full grid.